# Etiquetado de Sentimiento con Gemma
## Construcción del dataset de entrenamiento para fine-tuning de BETO

### Estrategia de etiquetado

1. **Comprensión**: Gemma lee el artículo completo y genera un análisis de contexto
2. **Etiquetado**: Gemma etiqueta cada oración usando ese análisis como referencia

### Clases
| Label | Descripción |
|---|---|
| **POS** | Expresa algo favorable dentro del encuadre de la noticia |
| **NEG** | Expresa algo desfavorable dentro del encuadre de la noticia |
| **NEU** | Descriptiva, sin carga valorativa — datos, cifras, definiciones |
| **MIXTO** | Combina señales positivas y negativas |

### Estrategia de batches
- **1 artículo por archivo** → 9 artículos por batch
- Tiempo estimado por batch: ~3 horas (20 min × 9 artículos)
- Sin overlap entre batches — cada artículo se etiqueta una sola vez
- Maximiza cobertura de todos los portales desde el primer batch

### Flujo para cada batch
1. Ejecutar celda **3** → selecciona los artículos
2. Ejecutar celda **8** → procesa y etiqueta
3. Ejecutar celdas **9** y **10** → revisa distribución y decide si continuar

## 1. Instalación de dependencias

In [8]:
# !pip install requests nltk pandas scikit-learn


## 2. Imports y configuración

In [9]:
import json
import re
import time
import random
import requests
from pathlib import Path
from collections import Counter
from datetime import datetime

import pandas as pd

# ── Rutas ──────────────────────────────────────────────────────
# tesis 4.0: ruta ajustada (en tesis 2.0 era "../../02_data_understanding/noticias/resultados")
BASE_DIR   = Path("../noticias/resultados")
OUTPUT_DIR = Path("dataset_etiquetado")
OUTPUT_DIR.mkdir(exist_ok=True)

ARCHIVOS = [
    "ambito_economia.json",
    "ambito_finanzas.json",
    "clarin_economia.json",
    "clarin_sitemap.json",
    "eleconomista_sitemap.json",
    "infobae_economia.json",
    "iprofesional_sitemap.json",
    "lanacion_economia.json",
    "tn_economia.json",
]

# ── Parámetros de batch ────────────────────────────────────────
# 1 artículo por archivo → 9 artículos totales por batch (~3 hs)
BATCH_SIZE = 1
SEED       = 42

# ── Oraciones ─────────────────────────────────────────────────
MIN_LARGO_ORACION = 30

# ── Ollama ─────────────────────────────────────────────────────
OLLAMA_URL = "http://localhost:11434/api/chat"
MODELO     = "gemma4:26b"
TIMEOUT    = 180   # 3 min — Gemma tarda ~2 min por oración
REINTENTOS = 3

# ── Estado ─────────────────────────────────────────────────────
INDICE_PATH     = OUTPUT_DIR / "indice_procesados.json"
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.jsonl"

print("Configuracion lista")
print(f"  Batch size: {BATCH_SIZE} articulo/archivo x {len(ARCHIVOS)} archivos = {BATCH_SIZE * len(ARCHIVOS)} articulos por batch")
print(f"  Timeout: {TIMEOUT}s")


Configuracion lista
  Batch size: 1 articulo/archivo x 9 archivos = 9 articulos por batch
  Timeout: 180s


## 3. Selección del batch

Selecciona 1 artículo por archivo que no haya sido procesado antes.

**Ejecutar al inicio de cada nuevo batch.**

In [13]:
def cargar_indice():
    if INDICE_PATH.exists():
        with open(INDICE_PATH, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()

def guardar_indice(procesadas):
    with open(INDICE_PATH, "w", encoding="utf-8") as f:
        json.dump(list(procesadas), f, ensure_ascii=False)

def seleccionar_batch(procesadas):
    """
    Selecciona BATCH_SIZE articulos por archivo no procesados.
    Seed fijo garantiza orden determinista sin overlap entre batches.
    """
    random.seed(SEED)
    batch = []
    for archivo in ARCHIVOS:
        ruta = BASE_DIR / archivo
        if not ruta.exists():
            print(f"  No encontrado: {archivo}")
            continue
        with open(ruta, "r", encoding="utf-8") as f:
            data = json.load(f)
        noticias = data["noticias"] if isinstance(data, dict) else data
        shuffled = noticias.copy()
        random.shuffle(shuffled)
        seleccionados = []
        for n in shuffled:
            if n.get("url") not in procesadas and n.get("contenido"):
                n["_archivo"] = archivo.replace(".json", "")
                seleccionados.append(n)
            if len(seleccionados) == BATCH_SIZE:
                break
        batch.extend(seleccionados)
        if seleccionados:
            titulo = (seleccionados[0].get("titulo") or "Sin titulo")[:55]
            print(f"  {archivo.replace('.json',''):30s} -> {titulo}")
        else:
            print(f"  {archivo.replace('.json',''):30s} -> (sin articulos disponibles)")
    return batch

# ── Estado actual ────────────────────────────────────────────
procesadas      = cargar_indice()
n_batch_actual  = len(procesadas) // (BATCH_SIZE * len(ARCHIVOS)) + 1
print(f"URLs ya procesadas: {len(procesadas)} -> Batch #{n_batch_actual}")
print()
print("Articulos seleccionados:")
batch_actual = seleccionar_batch(procesadas)
print(f"\nTotal: {len(batch_actual)} articulos")


URLs ya procesadas: 27 -> Batch #4

Articulos seleccionados:
  ambito_economia                -> Marina Dal Poggetto: "Cuando los ingresos de capitales 
  ambito_finanzas                -> Dólar en zona de definiciones: los tres escenarios que 
  clarin_economia                -> Comercios en crisis: dónde hay más locales vacíos, qué 
  clarin_sitemap                 -> Fuerte mensaje de Ford sobre su fábrica de  Pacheco: "S
  eleconomista_sitemap           -> Dólar hoy: a cuánto cotiza este viernes 19 de diciembre
  infobae_economia               -> Qué es un swap y cómo funciona: los detalles del esquem
  iprofesional_sitemap           -> La inflación de mayo arranca en cero, según primeras me
  lanacion_economia              -> Los tres grupos más poderosos del sector energético se 
  tn_economia                    -> Los departamentos usados ganan terreno en CABA: qué pas

Total: 9 articulos


## 4. División en oraciones

In [10]:
import nltk
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

def split_oraciones(texto):
    try:
        try:
            tokenizer = nltk.data.load("tokenizers/punkt_tab/Spanish.pickle")
        except Exception:
            tokenizer = nltk.data.load("tokenizers/punkt/Spanish.pickle")
        oraciones = tokenizer.tokenize(texto)
    except Exception:
        oraciones = re.split(r'(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑÜ"])', texto)
    resultado = []
    for o in oraciones:
        o = re.sub(r"\s+", " ", o.strip().replace("\n", " "))
        if len(o) >= MIN_LARGO_ORACION:
            resultado.append(o)
    return resultado

print("Tokenizador listo")


Tokenizador listo


## 5. Prompts

### Prompt 1 — Comprensión del artículo
Gemma lee el artículo y genera un análisis de contexto (tema, encuadre, actores, tono).

### Prompt 2 — Etiquetado por oración
Gemma etiqueta usando el análisis anterior como referencia, según lo que la oración expresa por sí misma.

In [11]:
PROMPT_COMPRENSION_SYSTEM = (
    "Sos un experto en analisis de noticias economicas argentinas.\n"
    "Vas a leer un articulo y generar un analisis de contexto que usaras como referencia\n"
    "para etiquetar el sentimiento de cada oracion.\n"
    "\n"
    "Responde UNICAMENTE con un JSON con exactamente estos campos, sin texto adicional:\n"
    '{'
    '"tema_central": "de que trata la noticia",'
    ' "encuadre": "perspectiva (mercados, consumidor, gobierno, sector, etc.)",'
    ' "actores_principales": "quienes son los protagonistas y su rol",'
    ' "tono_general": "tono predominante (optimista, alarmista, neutro, critico, etc.)"'
    '}'
)

PROMPT_COMPRENSION_USER = (
    "ARTICULO:\n"
    "Titulo: {titulo}\n"
    "Bajada: {bajada}\n"
    "\n"
    "{contenido}"
)

PROMPT_ETIQUETADO_SYSTEM = (
    "Sos un experto en analisis de sentimiento de noticias economicas argentinas.\n"
    "Previamente analizaste un articulo y generaste un contexto de referencia.\n"
    "\n"
    "Tu tarea es etiquetar el sentimiento de una oracion usando ese contexto.\n"
    "\n"
    "Reglas:\n"
    "- El sentimiento debe reflejarse en lo que la oracion expresa por si misma\n"
    "- Usa el contexto para resolver ambiguedades, no para inferir lo que la oracion no dice\n"
    "- Si la oracion es descriptiva o sin carga valorativa clara -> NEU\n"
    "- Si combina senales positivas y negativas de forma equilibrada -> MIXTO\n"
    "\n"
    "Clases:\n"
    "- POS: expresa algo favorable dentro del encuadre de la noticia\n"
    "- NEG: expresa algo desfavorable dentro del encuadre de la noticia\n"
    "- NEU: descriptiva, sin carga valorativa (datos, cifras, definiciones)\n"
    "- MIXTO: combina senales positivas y negativas\n"
    "\n"
    "Responde UNICAMENTE con un JSON con exactamente estos campos, sin texto adicional:\n"
    '{"label": "POS|NEG|NEU|MIXTO", "confidence": 0.0, "razon": "explicacion breve"}'
)

PROMPT_ETIQUETADO_USER = (
    "CONTEXTO DEL ARTICULO:\n"
    "{contexto}\n"
    "\n"
    "---\n"
    "ORACION A ETIQUETAR:\n"
    "\"{oracion}\""
)

print("Prompts definidos")


Prompts definidos


## 6. Cliente Ollama

In [12]:
def llamar_ollama(system, user):
    payload = {
        "model": MODELO,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "stream": False,
        "options": {"temperature": 0.1, "top_p": 0.9},
    }
    for intento in range(REINTENTOS):
        try:
            r = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
            r.raise_for_status()
            return r.json()["message"]["content"].strip()
        except Exception as e:
            print(f"    Intento {intento+1}/{REINTENTOS}: {e}")
            time.sleep(2 * (intento + 1))
    return None

def parsear_json(texto):
    if not texto:
        return None
    match = re.search(r'\{[^{}]+\}', texto, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return None

# Verificar conexion
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    modelos = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama conectado. Modelos: {modelos}")
    if not any(MODELO in m for m in modelos):
        print(f"ADVERTENCIA: {MODELO} no encontrado.")
except Exception as e:
    print(f"No se pudo conectar con Ollama: {e}")


Ollama conectado. Modelos: ['gemma4:26b-coding', 'qwen3.5:27b', 'gemma4:31b', 'gemma4:26b', 'qwen2.5:7b', 'qwen2.5:7b-instruct', 'llama3:latest', 'qwen3:30b', 'gpt-oss:20b', 'qwen3-coder:latest', 'deepseek-r1:32b', 'gemma3:27b-it-qat', 'devstral:latest', 'qwen3:latest', 'nomic-embed-text:latest', 'qwen2.5-coder:1.5b-base', 'qwen2.5-coder:14b', 'qwq:latest']


## 7. Test del pipeline (1 artículo, 3 oraciones)

Verificar que los prompts funcionan antes de correr el batch completo.

In [7]:
art_test = batch_actual[0]
print(f"Articulo: {art_test.get('titulo', 'Sin titulo')}")
print(f"Fuente:   {art_test['_archivo']}\n")

# Paso 1: Comprension
print("Paso 1: Comprension...")
resp_comp = llamar_ollama(
    PROMPT_COMPRENSION_SYSTEM,
    PROMPT_COMPRENSION_USER.format(
        titulo    = art_test.get("titulo") or "",
        bajada    = art_test.get("bajada") or "",
        contenido = (art_test.get("contenido") or "")[:3000],
    )
)
contexto_test = parsear_json(resp_comp)
print(json.dumps(contexto_test, ensure_ascii=False, indent=2))

# Paso 2: 3 oraciones de prueba
print("\nPaso 2: Etiquetado (3 oraciones de prueba)...")
oraciones_test = split_oraciones(art_test.get("contenido") or "")[:3]
contexto_str   = json.dumps(contexto_test, ensure_ascii=False)
for oracion in oraciones_test:
    resp = llamar_ollama(
        PROMPT_ETIQUETADO_SYSTEM,
        PROMPT_ETIQUETADO_USER.format(contexto=contexto_str, oracion=oracion)
    )
    resultado = parsear_json(resp)
    print(f"\n  {resultado}")
    print(f"  {oracion[:90]}...")


Articulo: El INDEC da a conocer la inflación de febrero: ¿habrá desaceleración por primera vez en nueve meses?
Fuente:   ambito_economia

Paso 1: Comprension...
{
  "tema_central": "Expectativas y debate técnico sobre la posible desaceleración de la inflación nacional en Argentina tras el dato de CABA.",
  "encuadre": "Macroeconómico y de mercado (análisis de indicadores de precios, proyecciones de consultoras y divergencia entre CABA y el resto del país).",
  "actores_principales": "INDEC (emisor de datos), economistas de consultoras (Rocío Bisang de GMA Capital, Gonzalo Carrera de Equilibra, Juan Manuel Telechea y Salvador Vitelli de Romano Group) y el Banco Central (vía REM).",
  "tono_general": "De incertidumbre y debate técnico, con una tensión entre la expectativa de una sorpresa a la baja y la cautela por la persistencia de ciertos rubros."
}

Paso 2: Etiquetado (3 oraciones de prueba)...

  {'label': 'POS', 'confidence': 0.95, 'razon': "La oración reporta una 'desaceleración' e

## 8. Procesamiento del batch

Procesa 1 artículo por portal (9 en total). El checkpoint guarda cada oración
en forma incremental — si se interrumpe, los datos no se pierden.

**Para un nuevo batch: ejecutar celda 3 y luego esta celda.**

In [19]:
MINIMO_POR_CLASE = 2500

def guardar_checkpoint(registro):
    with open(CHECKPOINT_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(registro, ensure_ascii=False) + '\n')

def cargar_checkpoint():
    registros = []
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
            for linea in f:
                try:
                    registros.append(json.loads(linea))
                except Exception:
                    pass
    return registros

def objetivo_alcanzado():
    registros = cargar_checkpoint()
    if not registros:
        return False
    conteo = pd.DataFrame(registros)['label'].value_counts()
    return all(conteo.get(l, 0) >= MINIMO_POR_CLASE for l in ['POS', 'NEG', 'NEU', 'MIXTO'])

def mostrar_distribucion():
    registros = cargar_checkpoint()
    if not registros:
        return
    df_tmp = pd.DataFrame(registros)
    conteo = df_tmp['label'].value_counts()
    pct    = df_tmp['label'].value_counts(normalize=True).mul(100).round(1)
    print(f"  Oraciones acumuladas: {len(df_tmp):,}")
    for label in ['POS', 'NEG', 'NEU', 'MIXTO']:
        n     = conteo.get(label, 0)
        p     = pct.get(label, 0.0)
        ok    = 'OK' if n >= MINIMO_POR_CLASE else f'faltan {MINIMO_POR_CLASE - n}'
        print(f"    {label:5s}: {n:5,} ({p:5.1f}%)  [{ok}]")

# Loop principal — corre hasta que todas las clases tengan 500 oraciones
n_batch = 0
while True:
    # Verificar objetivo antes de cada batch
    if objetivo_alcanzado():
        print('\n' + '='*60)
        print('OBJETIVO ALCANZADO: todas las clases tienen >= 500 oraciones.')
        print('Ejecuta la celda 11 para exportar el dataset.')
        print('='*60)
        mostrar_distribucion()
        break

    procesadas   = cargar_indice()
    n_batch      = len(procesadas) // (BATCH_SIZE * len(ARCHIVOS)) + 1
    batch_actual = seleccionar_batch(procesadas)

    if not batch_actual:
        print('Corpus agotado — no quedan articulos sin procesar.')
        mostrar_distribucion()
        break

    print(f"\n{'='*60}")
    print(f"BATCH #{n_batch} — {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*60}")

    urls_nuevas  = set()
    n_ok = n_err = 0
    inicio_batch = datetime.now()

    for i, art in enumerate(batch_actual):
        url    = art.get('url') or f'art_{i}'
        titulo = (art.get('titulo') or 'Sin titulo')[:80]
        fuente = art.get('_archivo', '')
        fecha  = (art.get('fecha_publicacion') or '')[:10]
        inicio_art = datetime.now()

        print(f"\n[{i+1}/{len(batch_actual)}] [{fuente}]")
        print(f"  {titulo}")

        oraciones = split_oraciones(art.get('contenido') or '')
        if not oraciones:
            print('  Sin oraciones validas, saltando.')
            urls_nuevas.add(url)
            continue

        # Paso 1: Comprension
        resp_comp = llamar_ollama(
            PROMPT_COMPRENSION_SYSTEM,
            PROMPT_COMPRENSION_USER.format(
                titulo    = art.get('titulo') or '',
                bajada    = art.get('bajada') or '',
                contenido = (art.get('contenido') or '')[:3000],
            )
        )
        contexto = parsear_json(resp_comp)
        if not contexto:
            print('  No se pudo generar contexto, saltando.')
            urls_nuevas.add(url)
            continue

        contexto_str = json.dumps(contexto, ensure_ascii=False)
        print(f"  Encuadre : {contexto.get('encuadre', '?')}")
        print(f"  Tono     : {contexto.get('tono_general', '?')}")
        print(f"  Oraciones: {len(oraciones)}")

        # Paso 2: Etiquetado
        for j, oracion in enumerate(oraciones):
            resp_etiq = llamar_ollama(
                PROMPT_ETIQUETADO_SYSTEM,
                PROMPT_ETIQUETADO_USER.format(contexto=contexto_str, oracion=oracion)
            )
            resultado = parsear_json(resp_etiq)
            if resultado and resultado.get('label') in {'POS', 'NEG', 'NEU', 'MIXTO'}:
                guardar_checkpoint({
                    'url'        : url,
                    'fuente'     : fuente,
                    'fecha'      : fecha,
                    'titulo'     : art.get('titulo') or '',
                    'or_idx'     : j,
                    'oracion'    : oracion,
                    'label'      : resultado['label'],
                    'confidence' : resultado.get('confidence'),
                    'razon'      : resultado.get('razon', ''),
                    'contexto'   : contexto,
                })
                n_ok += 1
                print(f"    [{j+1}/{len(oraciones)}] {resultado['label']:5s} ({resultado.get('confidence','?')}) — {oracion[:55]}...")
            else:
                n_err += 1
                print(f"    [{j+1}/{len(oraciones)}] Error — {oracion[:55]}...")

        urls_nuevas.add(url)
        mins = (datetime.now() - inicio_art).seconds // 60
        segs = (datetime.now() - inicio_art).seconds % 60
        print(f"  Completado en {mins}m {segs}s")

    # Actualizar indice
    procesadas.update(urls_nuevas)
    guardar_indice(procesadas)

    mins_total = (datetime.now() - inicio_batch).seconds // 60
    print(f"\nBatch #{n_batch} completado en {mins_total} min")
    print(f"  Articulos: {len(urls_nuevas)} | OK: {n_ok} | Errores: {n_err} | Acumulado: {len(procesadas)} arts")
    mostrar_distribucion()


  ambito_economia                -> Riesgo cambiario: el dólar quedó a tiro del techo de la
  ambito_finanzas                -> Dólar hoy: a cuánto cotiza este viernes 13 de febrero
  clarin_economia                -> Dólar tarjeta hoy: a cuánto cotiza este domingo 14 de d
  clarin_sitemap                 -> ARBA empezó a mandar las boletas del impuesto inmobilia
  eleconomista_sitemap           -> El mundo sigue inquieto y la Argentina continúa recuper
  infobae_economia               -> Una moda que sigue creciendo, cada vez con más modelos:
  iprofesional_sitemap           -> Caputo afronta un test decisivo para el dólar que despi
  lanacion_economia              -> “Perturbada”: anticipan cómo será la tormenta de Santa 
  tn_economia                    -> Cómo es la casa prefabricada que se vende por $465.000 

BATCH #32 — 23:45:39

[1/9] [ambito_economia]
  Riesgo cambiario: el dólar quedó a tiro del techo de la banda y la city ya mide 
  Encuadre : Macroeconómico y financiero (me

KeyboardInterrupt: 

## 9. Exportación del dataset final

Ejecutar cuando la distribución de clases sea satisfactoria.
Split estratificado 70 / 15 / 15 (train / validation / test).

In [ ]:
from sklearn.model_selection import train_test_split

df_ft = df[["oracion", "label", "fuente", "fecha", "url", "confidence", "razon"]].copy()

df_train, df_temp = train_test_split(df_ft, test_size=0.30, random_state=SEED, stratify=df_ft['label'])
df_val,   df_test = train_test_split(df_temp, test_size=0.50, random_state=SEED, stratify=df_temp['label'])

print(f"Train:      {len(df_train):,} oraciones")
print(f"Validation: {len(df_val):,} oraciones")
print(f"Test:       {len(df_test):,} oraciones")
print()

df_train.to_csv(OUTPUT_DIR / "train.csv",            index=False, encoding="utf-8")
df_val.to_csv(  OUTPUT_DIR / "validation.csv",       index=False, encoding="utf-8")
df_test.to_csv( OUTPUT_DIR / "test.csv",             index=False, encoding="utf-8")
df_ft.to_csv(   OUTPUT_DIR / "dataset_completo.csv", index=False, encoding="utf-8")

print("Dataset exportado:")
for fname in ["train.csv", "validation.csv", "test.csv", "dataset_completo.csv"]:
    print(f"  {OUTPUT_DIR}/{fname}")
